### RAG Pipeline - data ingestion to vector db pipeline

In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [14]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # add source information to meta data
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages ") 

        except Exception as e:
            print(f"Error : {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

    # process all PDFs in teh data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process.

Processing : DPI_Speech_Sound_Disorders9.261.pdf
Loaded 53 pages 

Processing : SLP-Medical-Review-Guidelines.pdf
Loaded 83 pages 

Total documents loaded: 136


In [15]:
all_pdf_documents

[Document(page_content='Speech Sound Disorders  \n \nSpeech Sound Assessment and Intervention Module  \n1 \n Table of Contents  \nIntroduction: Speech Sound Disorders …………………………………………………………………………………2  \nArticulation Disorders ………………………………………………………………………………………………….………3  \nPhonological Disorders ……………………………………………………………………………………………………..…3  \nAccented Speech ………………………………………………………………………………………………………………….4 \nTreatment of Speech Sound Disorders …………………………………………………………………………………4  \nI. Speech Referral Guidelines ……………………………………………………………………………….…5  \nI.a. Common Etiologies ……………………………………………………………….………………….5  \nI.b. Potential Consequenc es and Impact of Speech Impairment……………………..5  \nI.c. Major Milestones for Speech Development ………………………………………………6  \nI.d. Behaviors that Should Trigger an SLP Referral ………………………………………….6  \nI.e. World Health Organization Model (WHO) International …………………………..7  \nII. Screeni ng…………………………………………………………………………………………………………….9  \nII.a. ASHA Practice Policy ……………………………………………

In [16]:
### text splitting get into chunks

def split_documents(documents,chunk_size=1000 , chunk_overlap=200):
    """Spit documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n" , " " , ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documnets into {len(split_docs)} chunks")

    #show example of a chunk
    if  split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [17]:
chunks = split_documents(all_pdf_documents)
chunks

Split 136 documnets into 391 chunks

Example chunk:
Content: Speech Sound Disorders  
 
Speech Sound Assessment and Intervention Module  
1 
 Table of Contents  
Introduction: Speech Sound Disorders …………………………………………………………………………………2  
Articulation Disorders …………...
Metadata: {'source': '..\\data\\pdf\\DPI_Speech_Sound_Disorders9.261.pdf', 'page': 0, 'source_file': 'DPI_Speech_Sound_Disorders9.261.pdf', 'file_type': 'pdf'}


[Document(page_content='Speech Sound Disorders  \n \nSpeech Sound Assessment and Intervention Module  \n1 \n Table of Contents  \nIntroduction: Speech Sound Disorders …………………………………………………………………………………2  \nArticulation Disorders ………………………………………………………………………………………………….………3  \nPhonological Disorders ……………………………………………………………………………………………………..…3  \nAccented Speech ………………………………………………………………………………………………………………….4 \nTreatment of Speech Sound Disorders …………………………………………………………………………………4  \nI. Speech Referral Guidelines ……………………………………………………………………………….…5  \nI.a. Common Etiologies ……………………………………………………………….………………….5  \nI.b. Potential Consequenc es and Impact of Speech Impairment……………………..5  \nI.c. Major Milestones for Speech Development ………………………………………………6  \nI.d. Behaviors that Should Trigger an SLP Referral ………………………………………….6  \nI.e. World Health Organization Model (WHO) International …………………………..7  \nII. Screeni ng…………………………………………………………………………………………………………….9  \nII.a. ASHA Practice Policy ……………………………………………

### Embedding and vectorStoreDB


In [18]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [19]:

class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager 
        Args:
            model_name : Huggingface model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()  # call the internal method to load the model

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 

###Takes a list of text strings (texts) and turns each one into an embedding (a list of numbers).
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts : List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# Initialize Embedding Manager
embedding_manager = EmbeddingManager()


Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


VectorDB

In [20]:
#store embeddings of documents and retrieve them later.
class VectorStore:
    """Handles storage and retrieval of document embeddings using ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        """
        Initialize the VectorStore
            Args:
                collection_name : Name of the ChromaDB collection
                persist_directory : Directory to persist the vector store
            """
        self.collection_name = collection_name    #name for the database collection to store embeddings.
        self.persist_directory = persist_directory  #folder on your computer to save embeddings
        self.client = None     #hold the connection to ChromaDB.
        self.collection = None    #hold the actual collection of vectors.
        self._initialize_store()

    def _initialize_store(self):
        #This method sets up ChromaDB so you can store and retrieve embeddings for your documents.
        """Initialize ChromaDB client and collection"""
        try:
            # create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
              

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF Document Embeddings for RAG"})
            print(f"Vector store initialized,Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise    

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        Args:
           documents : List of langchain documents
           embeddings :  embeddings corresponding to documents
        """

        if len(documents) != len(embeddings):
                raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store")

        #  prepare data for ChromoDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata 
            metadata = dict(doc.metadata)  # copy existing metadata
            metadata['doc_index'] = i  # add index info
            metadata['content_length'] = len(doc.page_content)  # add content length
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

        # add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection now: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore=VectorStore()
vectorstore
    
    

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store initialized,Collection: pdf_documents
Existing documents in collection: 0


In [21]:
chunks

[Document(page_content='Speech Sound Disorders  \n \nSpeech Sound Assessment and Intervention Module  \n1 \n Table of Contents  \nIntroduction: Speech Sound Disorders …………………………………………………………………………………2  \nArticulation Disorders ………………………………………………………………………………………………….………3  \nPhonological Disorders ……………………………………………………………………………………………………..…3  \nAccented Speech ………………………………………………………………………………………………………………….4 \nTreatment of Speech Sound Disorders …………………………………………………………………………………4  \nI. Speech Referral Guidelines ……………………………………………………………………………….…5  \nI.a. Common Etiologies ……………………………………………………………….………………….5  \nI.b. Potential Consequenc es and Impact of Speech Impairment……………………..5  \nI.c. Major Milestones for Speech Development ………………………………………………6  \nI.d. Behaviors that Should Trigger an SLP Referral ………………………………………….6  \nI.e. World Health Organization Model (WHO) International …………………………..7  \nII. Screeni ng…………………………………………………………………………………………………………….9  \nII.a. ASHA Practice Policy ……………………………………………

In [24]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks] 
texts

## generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#### Add documents and embeddings to vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 391 texts


Batches: 100%|██████████| 13/13 [00:46<00:00,  3.55s/it]


Generated embeddings with shape: (391, 384)
Adding 391 documents to vector store


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Successfully added 391 documents to vector store.
Total documents in collection now: 391
